# RSI (Relative Strength Index) Strategy Notebook

**Framework:** Rick's FTMO Strategy Testing Template  
**Indicator:** RSI — Classic mean-reversion / momentum oscillator  
**Purpose:** Extensive grid search optimization with IS/OOS validation and sensitivity analysis  

---

### Strategy Logic (Mean-Reversion)
- **LONG Entry:** RSI crosses back ABOVE the oversold threshold (bounce from oversold)
- **LONG Exit:** RSI crosses back BELOW the overbought threshold (rejection from overbought)

### Grid Search — EXTENSIVE
- RSI Period: 5–30 (step 1) → **26 values**
- Oversold Threshold: 15–40 (step 1) → **26 values**
- Overbought Threshold: 60–85 (step 1) → **26 values**
- Filter: oversold < 50 < overbought
- **~17,000+ valid combinations**

In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

---
## Section 1: Data Loading

In [ ]:
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'BTC-USD'  # Any ticker symbol (e.g., 'EURUSD=X', 'GC=F', '^GSPC')
START_DATE = '2018-01-01'  # Any start date in YYYY-MM-DD format

# Download data from start date onwards
try:
    stock_data = yf.download(TICKER, start=START_DATE, interval='1d')
except Exception as e:
    print(f"Error downloading data: {e}")
    stock_data = pd.DataFrame()

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")

stock_data

---
## Section 2: Indicator Calculation — RSI

In [ ]:
# RSI INDICATOR CALCULATION (default period=14 for display)

if "stock_data" not in locals() or stock_data.empty:
    raise ValueError("Please run the stock data download cell first")

# Extract OHLCV data (handling multi-level columns from yfinance)
if isinstance(stock_data.columns, pd.MultiIndex):
    close_arr = stock_data[("Close", TICKER)].values
    high_arr = stock_data[("High", TICKER)].values
    low_arr = stock_data[("Low", TICKER)].values
    volume_arr = stock_data[("Volume", TICKER)].values
else:
    close_arr = stock_data["Close"].values
    high_arr = stock_data["High"].values
    low_arr = stock_data["Low"].values
    volume_arr = stock_data["Volume"].values

print(f"Calculating RSI indicators for {TICKER}...")

# RSI with common periods for display
rsi_7 = talib.RSI(close_arr, timeperiod=7)
rsi_14 = talib.RSI(close_arr, timeperiod=14)
rsi_21 = talib.RSI(close_arr, timeperiod=21)

# Reference MAs
sma_20 = talib.SMA(close_arr, timeperiod=20)
sma_50 = talib.SMA(close_arr, timeperiod=50)

# Build indicators dataframe
indicators_df = pd.DataFrame({
    "Date": stock_data.index,
    "Close": close_arr,
    "SMA_20": sma_20,
    "SMA_50": sma_50,
    "RSI_7": rsi_7,
    "RSI_14": rsi_14,
    "RSI_21": rsi_21
})

print("All indicators calculated!")
print(f"Data shape: {indicators_df.shape}")
indicators_df.tail(5)

---
## Section 3: Signal Generation & Visualization

In [ ]:
# VISUALIZE RSI SIGNALS (default RSI 14, OS=30, OB=70)

rsi_s = pd.Series(rsi_14, index=stock_data.index)
close_s = pd.Series(close_arr, index=stock_data.index)

# Mean-reversion signals:
# Entry: RSI crosses back ABOVE oversold (30)
# Exit:  RSI crosses back BELOW overbought (70)
entry_signals = (rsi_s.shift(1) <= 30) & (rsi_s > 30)
exit_signals  = (rsi_s.shift(1) >= 70) & (rsi_s < 70)

print(f"Default RSI(14, OS=30, OB=70) Signals:")
print(f"  Total Entry Signals: {entry_signals.sum()}")
print(f"  Total Exit Signals:  {exit_signals.sum()}")

# 2-panel chart
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

# Price with entry/exit markers
ax1.plot(stock_data.index, close_arr, color='black', linewidth=1, label='Close')
ax1.scatter(stock_data.index[entry_signals.fillna(False).values],
            close_s[entry_signals.fillna(False)],
            marker='^', color='green', s=70, zorder=5, label='Long Entry (RSI > OS)')
ax1.scatter(stock_data.index[exit_signals.fillna(False).values],
            close_s[exit_signals.fillna(False)],
            marker='v', color='red', s=70, zorder=5, label='Long Exit (RSI < OB)')
ax1.set_title(f'{TICKER} Price with RSI(14) Mean-Reversion Signals', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# RSI panel
ax2.plot(stock_data.index, rsi_14, color='purple', linewidth=1, label='RSI(14)')
ax2.axhline(70, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Overbought (70)')
ax2.axhline(30, color='green', linestyle='--', linewidth=1, alpha=0.7, label='Oversold (30)')
ax2.axhline(50, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
ax2.fill_between(stock_data.index, 70, 100, alpha=0.1, color='red')
ax2.fill_between(stock_data.index, 0, 30, alpha=0.1, color='green')
ax2.set_ylim(0, 100)
ax2.set_title('RSI(14)', fontsize=12, fontweight='bold')
ax2.set_ylabel('RSI', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 4: Prepare Price Series & Train/Validation Split

In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

def select_close_series(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ('Close', ticker) in df.columns:
            s = df[('Close', ticker)]
        else:
            cols = [c for c in df.columns if 'Close' in str(c)]
            if not cols:
                raise KeyError("Close not found")
            s = df[cols[0]]
    else:
        s = df['Close']
    return s.astype(float).squeeze()

close = select_close_series(stock_data, TICKER)
close.name = 'price'

# Simple 60/40 split
TRAIN_RATIO = 0.60
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close   = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} → {train_close.index[-1].date()} | "
      f"val={val_close.index[0].date()} → {val_close.index[-1].date()}")
print(f"Train samples: {len(train_close)} | Val samples: {len(val_close)}")

---
## Section 5: RSI Grid Search — EXTENSIVE Parameter Optimization

### Parameter Ranges (Dense Coverage)
- **rsi_period:** 5 to 30, step 1 → **26 values**
- **oversold:** 15 to 40, step 1 → **26 values**
- **overbought:** 60 to 85, step 1 → **26 values**
- Filter: oversold < 50, overbought > 50
- Expected: **~17,576 valid combinations**

### Signal Logic (Mean-Reversion)
- Entry: RSI crosses back above oversold threshold
- Exit: RSI crosses back below overbought threshold

### Cost Model
- Fees: 0.05% (matching TEMA template)
- Slippage: 0.05%

---

In [ ]:
# DEFINE EXTENSIVE PARAMETER RANGES

rsi_periods = list(range(5, 31, 1))        # 26 values: 5,6,...,30
oversold_levels = list(range(15, 41, 1))    # 26 values: 15,16,...,40
overbought_levels = list(range(60, 86, 1))  # 26 values: 60,61,...,85

# All combinations are valid (oversold < 50 < overbought guaranteed by ranges)
rsi_combinations = list(product(rsi_periods, oversold_levels, overbought_levels))

print(f"RSI Parameter Ranges:")
print(f"  RSI periods ({len(rsi_periods)}):       {rsi_periods[0]}–{rsi_periods[-1]}")
print(f"  Oversold levels ({len(oversold_levels)}):   {oversold_levels[0]}–{oversold_levels[-1]}")
print(f"  Overbought levels ({len(overbought_levels)}): {overbought_levels[0]}–{overbought_levels[-1]}")
print(f"\nGenerated {len(rsi_combinations)} valid RSI combinations")
print(f"\n📋 First 10 combinations preview:")
for i, (p, os, ob) in enumerate(rsi_combinations[:10], 1):
    print(f"  {i:2d}. Period:{p:2d} | Oversold:{os:2d} | Overbought:{ob:2d}")
if len(rsi_combinations) > 10:
    print(f"   ... and {len(rsi_combinations) - 10} more combinations")
print("\nReady to test all combinations on training data!")

In [ ]:
# INITIALIZE RESULTS COLLECTION

grid_search_results = []

print("RSI Results Collection System Initialized")
print(f"   - Will test {len(rsi_combinations)} RSI combinations")
print(f"   - Results will be stored in 'grid_search_results' list")

metrics_to_collect = [
    "rsi_period", "oversold", "overbought",
    "total_return", "annualized_return",
    "sharpe_ratio", "sortino_ratio",
    "max_drawdown", "volatility", "ulcer_index",
    "win_rate", "total_trades", "avg_trade_duration",
    "expectancy", "profit_factor", "payoff_ratio",
    "avg_win_amount", "avg_loss_amount",
    "trades_per_year"
]

print(f"\nMetrics to collect for each RSI combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")
print("\nReady to start the RSI grid search!")

In [ ]:
# HELPER: Compute RSI entry/exit signals for given parameters

def compute_rsi_signals(price_series, rsi_period, oversold, overbought):
    """
    Compute RSI mean-reversion signals.
    
    Entry: RSI crosses back ABOVE oversold threshold (bounce)
    Exit:  RSI crosses back BELOW overbought threshold (rejection)
    
    Returns:
        entries (pd.Series[bool]): shifted by 1 bar to avoid lookahead
        exits   (pd.Series[bool]): shifted by 1 bar to avoid lookahead
    """
    arr = price_series.values.astype(float)
    rsi = talib.RSI(arr, timeperiod=rsi_period)
    rsi_s = pd.Series(rsi, index=price_series.index)
    
    # Mean reversion: buy the oversold bounce, exit the overbought rejection
    entries_raw = (rsi_s.shift(1) <= oversold) & (rsi_s > oversold)
    exits_raw   = (rsi_s.shift(1) >= overbought) & (rsi_s < overbought)
    
    # Shift by 1 bar to avoid lookahead bias
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits   = exits_raw.shift(1).fillna(False).astype(bool)
    
    return entries, exits

print("✓ RSI signal computation function defined.")

In [ ]:
# VISUALIZE SIGNALS FOR ONE EXAMPLE BEFORE GRID SEARCH

ex_period, ex_os, ex_ob = 14, 30, 70
entries_ex, exits_ex = compute_rsi_signals(train_close, ex_period, ex_os, ex_ob)

arr_ex = train_close.values.astype(float)
rsi_ex = talib.RSI(arr_ex, timeperiod=ex_period)

signals_df = pd.DataFrame({
    'Close': train_close.values,
    'RSI': rsi_ex,
    'Buy': entries_ex.values,
    'Sell': exits_ex.values
}, index=train_close.index)
signals_df.index.name = 'Date'

pos = 0
pos_list = []
for buy, sell in zip(signals_df['Buy'], signals_df['Sell']):
    if buy: pos = 1
    elif sell: pos = 0
    pos_list.append(pos)
signals_df['Position'] = pos_list

print(f"Example RSI({ex_period}, OS={ex_os}, OB={ex_ob}) on training data:")
print(f"  Entry signals: {signals_df['Buy'].sum()}")
print(f"  Exit signals:  {signals_df['Sell'].sum()}")
print(f"\nRows with signals:")
signal_rows = signals_df[(signals_df['Buy']) | (signals_df['Sell'])]
print(signal_rows.head(20).to_string())

In [ ]:
# RSI GRID SEARCH ON TRAINING DATA — VECTORIZED & BATCHED

print("INITIATING VECTORIZED RSI GRID SEARCH OPTIMIZATION")
print("=" * 70)
print(f"Testing Strategy: RSI Mean-Reversion (Oversold Bounce / Overbought Exit)")
print(f"Training Period: {train_close.index[0].date()} → {train_close.index[-1].date()}")
print(f"Initial Capital: $100,000")
print(f"Transaction Costs: 0.05% fees + 0.05% slippage per trade")
print(f"Optimization Metric: Sharpe Ratio (risk-adjusted returns)")
print("=" * 70)

# Pre-compute ALL unique RSI series to avoid redundant TA-Lib calls
# (same RSI period with different thresholds = same RSI series)
print("\nPre-computing RSI series for all unique periods...")
rsi_cache = {}
train_arr = train_close.values.astype(float)
for period in rsi_periods:
    rsi_vals = talib.RSI(train_arr, timeperiod=period)
    rsi_cache[period] = pd.Series(rsi_vals, index=train_close.index)
print(f"✓ Pre-computed {len(rsi_cache)} unique RSI series\n")

# Configuration
BATCH_SIZE = 1000
total_combinations = len(rsi_combinations)

print(f"Total combinations to test: {total_combinations}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Using pre-computed RSI + vectorized backtesting\n")

grid_search_results = []
successful_tests = 0
failed_tests = 0

print("Starting batched grid search...\n")

for batch_start in range(0, total_combinations, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total_combinations)
    batch_combos = rsi_combinations[batch_start:batch_end]
    batch_size = len(batch_combos)
    
    batch_num = batch_start // BATCH_SIZE + 1
    total_batches = (total_combinations + BATCH_SIZE - 1) // BATCH_SIZE
    
    # Build signal matrices using pre-computed RSI (FAST)
    batch_entries = []
    batch_exits = []
    
    for period, oversold, overbought in batch_combos:
        try:
            rsi_s = rsi_cache[period]
            entries_raw = (rsi_s.shift(1) <= oversold) & (rsi_s > oversold)
            exits_raw   = (rsi_s.shift(1) >= overbought) & (rsi_s < overbought)
            entries = entries_raw.shift(1).fillna(False).astype(bool)
            exits   = exits_raw.shift(1).fillna(False).astype(bool)
            batch_entries.append(entries)
            batch_exits.append(exits)
        except Exception:
            batch_entries.append(pd.Series(False, index=train_close.index, dtype=bool))
            batch_exits.append(pd.Series(False, index=train_close.index, dtype=bool))
            failed_tests += 1
    
    # Convert to DataFrames
    entries_df = pd.DataFrame(batch_entries).T
    entries_df.index = train_close.index
    exits_df = pd.DataFrame(batch_exits).T
    exits_df.index = train_close.index
    
    # Vectorized backtest
    try:
        portfolios = vbt.Portfolio.from_signals(
            close=train_close,
            entries=entries_df,
            exits=exits_df,
            init_cash=100_000,
            fees=0.0005,
            slippage=0.0005,
            freq='D'
        )
        
        total_returns = portfolios.total_return()
        annualized_returns = portfolios.annualized_return(freq='D')
        max_drawdowns = portfolios.max_drawdown()
        volatilities = portfolios.annualized_volatility(freq='D')
        sharpe_ratios = portfolios.sharpe_ratio(freq='D')
        sortino_ratios = portfolios.sortino_ratio(freq='D')
        
        for idx, (period, oversold, overbought) in enumerate(batch_combos):
            try:
                total_return = float(total_returns.iloc[idx] if hasattr(total_returns, 'iloc') else total_returns)
                annualized_return = float(annualized_returns.iloc[idx] if hasattr(annualized_returns, 'iloc') else annualized_returns)
                max_drawdown = float(max_drawdowns.iloc[idx] if hasattr(max_drawdowns, 'iloc') else max_drawdowns)
                volatility = float(volatilities.iloc[idx] if hasattr(volatilities, 'iloc') else volatilities)
                sharpe_ratio = float(sharpe_ratios.iloc[idx] if hasattr(sharpe_ratios, 'iloc') else sharpe_ratios)
                sortino_ratio = float(sortino_ratios.iloc[idx] if hasattr(sortino_ratios, 'iloc') else sortino_ratios)
                
                trades = portfolios[idx].trades if batch_size > 1 else portfolios.trades
                total_trades = len(trades)
                
                years = max((train_close.index[-1] - train_close.index[0]).days / 365.25, 1e-9)
                trades_per_year = total_trades / years
                
                if trades_per_year < 2:
                    continue
                
                win_rate_pct = np.nan
                profit_factor = np.nan
                expectancy = 0.0
                avg_win_amount = 0.0
                avg_loss_amount = 0.0
                payoff_ratio = np.nan
                avg_duration = np.nan
                
                if total_trades > 0:
                    tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
                    if tr.size > 0:
                        pos = tr[tr > 0]
                        neg = tr[tr < 0]
                        win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
                        gains = pos.sum() if len(pos) else 0.0
                        losses = abs(neg.sum()) if len(neg) else 0.0
                        profit_factor = gains / losses if losses > 0 else np.inf
                        expectancy = float(tr.mean())
                        avg_win_amount = float(pos.mean()) if len(pos) else 0.0
                        avg_loss_amount = float(abs(neg.mean())) if len(neg) else 0.0
                        payoff_ratio = (avg_win_amount / avg_loss_amount) if avg_loss_amount > 0 else np.inf
                    
                    try:
                        durations = trades.duration
                        if hasattr(durations, 'mean'):
                            d_mean = durations.mean()
                            avg_duration = float(d_mean.total_seconds() / 86400) if hasattr(d_mean, 'total_seconds') else float(d_mean)
                    except:
                        pass
                
                try:
                    returns_series = portfolios.returns().iloc[:, idx] if batch_size > 1 else portfolios.returns()
                    cum = (1 + returns_series).cumprod()
                    peak = cum.cummax()
                    dd = (cum - peak) / peak
                    ulcer_index = float(np.sqrt((dd.pow(2)).mean()))
                except:
                    ulcer_index = np.nan
                
                grid_search_results.append({
                    "rsi_period": period,
                    "oversold": oversold,
                    "overbought": overbought,
                    "total_return": total_return,
                    "annualized_return": annualized_return,
                    "max_drawdown": max_drawdown,
                    "volatility": volatility,
                    "sharpe_ratio": sharpe_ratio,
                    "sortino_ratio": sortino_ratio,
                    "ulcer_index": ulcer_index,
                    "total_trades": total_trades,
                    "win_rate": win_rate_pct,
                    "profit_factor": profit_factor,
                    "expectancy": expectancy,
                    "avg_win_amount": avg_win_amount,
                    "avg_loss_amount": avg_loss_amount,
                    "payoff_ratio": payoff_ratio,
                    "avg_trade_duration": avg_duration,
                    "trades_per_year": trades_per_year
                })
                successful_tests += 1
                
            except Exception:
                failed_tests += 1
    
    except Exception as e:
        print(f"  ⚠️ Batch failed: {str(e)[:100]}")
        failed_tests += batch_size
    
    progress_pct = (batch_end / total_combinations) * 100
    if batch_num % 2 == 0 or batch_num == total_batches:
        print(f"  ✓ Batch {batch_num}/{total_batches} | Progress: {batch_end}/{total_combinations} "
              f"({progress_pct:.1f}%) | Successful: {successful_tests}")

# SUMMARY
print("\n" + "=" * 70)
print("VECTORIZED GRID SEARCH COMPLETED!")
print("=" * 70)
print(f"Total combinations attempted: {total_combinations}")
print(f"Successfully completed: {successful_tests}")
print(f"Failed/filtered: {total_combinations - successful_tests}")
print(f"Success rate: {(successful_tests/total_combinations)*100:.1f}%")
print(f"\n✓ Results stored in 'grid_search_results' ({len(grid_search_results)} entries)")

if successful_tests > 0:
    results_df_preview = pd.DataFrame(grid_search_results)
    
    print("\n" + "=" * 70)
    print("🏆 TOP 5 COMBINATIONS (by In-Sample Sharpe Ratio)")
    print("=" * 70)
    
    top_5 = results_df_preview.nlargest(5, 'sharpe_ratio')
    for rank, (idx, row) in enumerate(top_5.iterrows(), 1):
        print(f"\n#{rank} - RSI({int(row['rsi_period'])}, OS={int(row['oversold'])}, OB={int(row['overbought'])})")
        print(f"   Sharpe Ratio:      {row['sharpe_ratio']:.3f}")
        print(f"   Total Return:      {row['total_return']:.2%}")
        print(f"   Annualized Return: {row['annualized_return']:.2%}")
        print(f"   Max Drawdown:      {row['max_drawdown']:.2%}")
        print(f"   Win Rate:          {row['win_rate']:.1f}%")
        print(f"   Profit Factor:     {row['profit_factor']:.2f}")
        print(f"   Total Trades:      {int(row['total_trades'])} ({row['trades_per_year']:.1f}/year)")
    
    print("\n" + "=" * 70)

---
## Section 6: Results Display & Analysis

In [ ]:
# GRID SEARCH RESULTS ANALYSIS

results_df = pd.DataFrame(grid_search_results)

print("Grid Search Results Analysis")
print("=" * 50)
print(f"Total combinations tested: {len(results_df)}")
print(f"Results shape: {results_df.shape}")

print("\nComprehensive Performance Statistics:")
print("-" * 50)
print("Return Metrics:")
print(f"   Best Total Return: {results_df['total_return'].max():.2%}")
print(f"   Average Total Return: {results_df['total_return'].mean():.2%}")
print(f"   Best Annualized Return: {results_df['annualized_return'].max():.2%}")
print("Risk-Adjusted Return Metrics:")
print(f"   Best Sharpe Ratio: {results_df['sharpe_ratio'].max():.3f}")
print(f"   Median Sharpe Ratio: {results_df['sharpe_ratio'].median():.3f}")
print(f"   75th Percentile Sharpe: {results_df['sharpe_ratio'].quantile(0.75):.3f}")
print(f"   Best Sortino Ratio: {results_df['sortino_ratio'].max():.3f}")
print("Risk Metrics:")
print(f"   Average Max Drawdown: {results_df['max_drawdown'].mean():.2%}")
print(f"   Best (smallest) Max DD: {results_df['max_drawdown'].max():.2%}")
print("Trade Performance:")
print(f"   Best Win Rate: {results_df['win_rate'].max():.1f}%")
print(f"   Average Win Rate: {results_df['win_rate'].mean():.1f}%")
print(f"   Best Profit Factor: {results_df['profit_factor'].replace([np.inf], np.nan).max():.2f}")
print(f"   Avg Trades/Year: {results_df['trades_per_year'].mean():.1f}")

# Sharpe distribution
print("\nSharpe Ratio Distribution:")
for threshold in [0.5, 1.0, 1.5, 2.0]:
    count = (results_df['sharpe_ratio'] >= threshold).sum()
    pct = count / len(results_df) * 100
    print(f"   Sharpe >= {threshold:.1f}: {count:4d} combos ({pct:.1f}%)")

# Display top 50
print("\n" + "=" * 90)
print("TOP 50 PARAMETER COMBINATIONS (sorted by IS Sharpe Ratio)")
print("=" * 90)

display_cols = ['rsi_period', 'oversold', 'overbought',
                'sharpe_ratio', 'total_return', 'annualized_return', 'max_drawdown',
                'win_rate', 'profit_factor', 'total_trades', 'trades_per_year']
top_50 = results_df.nlargest(50, 'sharpe_ratio')[display_cols].copy()
top_50['total_return'] = top_50['total_return'].apply(lambda x: f"{x:.2%}")
top_50['annualized_return'] = top_50['annualized_return'].apply(lambda x: f"{x:.2%}")
top_50['max_drawdown'] = top_50['max_drawdown'].apply(lambda x: f"{x:.2%}")
top_50['sharpe_ratio'] = top_50['sharpe_ratio'].apply(lambda x: f"{x:.3f}")
top_50['win_rate'] = top_50['win_rate'].apply(lambda x: f"{x:.1f}%")
top_50['profit_factor'] = top_50['profit_factor'].apply(lambda x: f"{x:.2f}")
print(top_50.to_string(index=False))

---
## Section 7: Out-of-Sample Validation — IS vs OOS Comparison

In [ ]:
# OUT-OF-SAMPLE VALIDATION FOR TOP COMBINATIONS

results_df_raw = pd.DataFrame(grid_search_results)
top_n = min(50, len(results_df_raw))
top_is = results_df_raw.nlargest(top_n, 'sharpe_ratio').copy()

oos_results = []

print(f"Validating top {top_n} IS combinations on OOS data...")
print(f"OOS Period: {val_close.index[0].date()} → {val_close.index[-1].date()}\n")

for _, row in top_is.iterrows():
    period = int(row['rsi_period'])
    oversold = int(row['oversold'])
    overbought = int(row['overbought'])
    
    try:
        entries_oos, exits_oos = compute_rsi_signals(val_close, period, oversold, overbought)
        
        pf_oos = vbt.Portfolio.from_signals(
            close=val_close, entries=entries_oos, exits=exits_oos,
            init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D'
        )
        
        oos_sharpe = float(pf_oos.sharpe_ratio(freq='D'))
        oos_return = float(pf_oos.total_return())
        oos_maxdd = float(pf_oos.max_drawdown())
        oos_trades = len(pf_oos.trades)
        
        oos_win_rate = np.nan
        oos_profit_factor = np.nan
        if oos_trades > 0:
            tr = pf_oos.trades.returns.values
            pos_t = tr[tr > 0]
            neg_t = tr[tr < 0]
            oos_win_rate = (len(pos_t) / len(tr)) * 100.0
            gains = pos_t.sum() if len(pos_t) else 0.0
            losses = abs(neg_t.sum()) if len(neg_t) else 0.0
            oos_profit_factor = gains / losses if losses > 0 else np.inf
        
        oos_results.append({
            'rsi_period': period, 'oversold': oversold, 'overbought': overbought,
            'is_sharpe': float(row['sharpe_ratio']),
            'is_return': float(row['total_return']),
            'is_maxdd': float(row['max_drawdown']),
            'is_trades': int(row['total_trades']),
            'oos_sharpe': oos_sharpe,
            'oos_return': oos_return,
            'oos_maxdd': oos_maxdd,
            'oos_trades': oos_trades
        })
    except:
        pass

oos_df = pd.DataFrame(oos_results).sort_values('is_sharpe', ascending=False)

print("\n" + "=" * 100)
print("IS vs OOS COMPARISON TABLE")
print("=" * 100)
print("Interpretation: OOS Sharpe within 30% of IS Sharpe = ROBUST\n")

display_oos = oos_df.copy()
display_oos['sharpe_decay'] = ((display_oos['oos_sharpe'] - display_oos['is_sharpe']) / 
                                display_oos['is_sharpe'].abs() * 100)
display_oos['robust'] = display_oos['sharpe_decay'].abs() < 30

for col in ['is_sharpe', 'oos_sharpe']:
    display_oos[col] = display_oos[col].apply(lambda x: f"{x:.3f}")
for col in ['is_return', 'oos_return', 'is_maxdd', 'oos_maxdd']:
    display_oos[col] = display_oos[col].apply(lambda x: f"{x:.2%}")
display_oos['sharpe_decay'] = display_oos['sharpe_decay'].apply(lambda x: f"{x:.1f}%")
display_oos['robust'] = display_oos['robust'].apply(lambda x: '✅' if x else '❌')

print(display_oos[['rsi_period', 'oversold', 'overbought',
                    'is_sharpe', 'is_return', 'is_maxdd',
                    'oos_sharpe', 'oos_return', 'oos_maxdd', 'oos_trades',
                    'sharpe_decay', 'robust']].head(30).to_string(index=False))

robust_count = sum(1 for r in oos_results 
                   if abs((r['oos_sharpe'] - r['is_sharpe']) / max(abs(r['is_sharpe']), 1e-9)) < 0.3)
print(f"\n✅ Robust combinations (OOS within 30% of IS): {robust_count}/{len(oos_results)}")

---
## Section 8: Sensitivity Analysis (CRITICAL — Rick's Framework)

6-panel bar chart: 3 params × IS/OOS

In [ ]:
# SENSITIVITY ANALYSIS — RICK'S FRAMEWORK

results_df = pd.DataFrame(grid_search_results)
FREQ = '1D'

best_row = results_df.loc[results_df['sharpe_ratio'].idxmax()]
period_base = int(best_row['rsi_period'])
os_base = int(best_row['oversold'])
ob_base = int(best_row['overbought'])
base_is_sharpe = float(best_row['sharpe_ratio'])

print(f"📊 BASE PARAMETERS (Best IS Sharpe):")
print(f"   RSI({period_base}, OS={os_base}, OB={ob_base})")
print(f"   IS Sharpe:  {base_is_sharpe:.3f}")
print(f"   IS Return:  {best_row['total_return']:.2%}")
print(f"   IS MaxDD:   {best_row['max_drawdown']:.2%}")

# Generate variation ranges (±5 around base)
period_range = list(range(max(3, period_base - 5), period_base + 6))
os_range = list(range(max(5, os_base - 5), min(49, os_base + 6)))
ob_range = list(range(max(51, ob_base - 5), min(96, ob_base + 6)))

print(f"\n📐 Variation Ranges:")
print(f"   Period:     {period_range}")
print(f"   Oversold:   {os_range}")
print(f"   Overbought: {ob_range}")

# Build all sensitivity combos
all_combos = set()
for p in period_range:
    all_combos.add((p, os_base, ob_base))
for os_val in os_range:
    all_combos.add((period_base, os_val, ob_base))
for ob_val in ob_range:
    all_combos.add((period_base, os_base, ob_val))
all_combos = sorted(list(all_combos))

def eval_combo_both(period, oversold, overbought):
    entries_is, exits_is = compute_rsi_signals(train_close, period, oversold, overbought)
    pf_is = vbt.Portfolio.from_signals(
        close=train_close, entries=entries_is, exits=exits_is,
        init_cash=100_000, fees=0.0005, slippage=0.0005, freq=FREQ
    )
    entries_oos, exits_oos = compute_rsi_signals(val_close, period, oversold, overbought)
    pf_oos = vbt.Portfolio.from_signals(
        close=val_close, entries=entries_oos, exits=exits_oos,
        init_cash=100_000, fees=0.0005, slippage=0.0005, freq=FREQ
    )
    return {
        'period': period, 'oversold': oversold, 'overbought': overbought,
        'is_sharpe': float(pf_is.sharpe_ratio(freq='D')),
        'is_return': float(pf_is.total_return()),
        'is_maxdd': float(pf_is.max_drawdown()),
        'oos_sharpe': float(pf_oos.sharpe_ratio(freq='D')),
        'oos_return': float(pf_oos.total_return()),
        'oos_maxdd': float(pf_oos.max_drawdown()),
        'oos_trades': len(pf_oos.trades)
    }

print(f"\n🔄 Testing {len(all_combos)} parameter variations...")
rows = []
for combo in all_combos:
    try:
        rows.append(eval_combo_both(*combo))
    except:
        pass

if not rows:
    print("❌ No sensitivity results computed.")
else:
    sens_df = pd.DataFrame(rows)
    
    base_oos_row = sens_df[(sens_df['period'] == period_base) & (sens_df['oversold'] == os_base) &
                           (sens_df['overbought'] == ob_base)]
    base_oos_sharpe = float(base_oos_row['oos_sharpe'].iloc[0]) if len(base_oos_row) > 0 else np.nan
    
    print(f"   Base IS Sharpe:  {base_is_sharpe:.3f}")
    print(f"   Base OOS Sharpe: {base_oos_sharpe:.3f}")
    
    # Split by parameter variation type
    period_variations = sens_df[(sens_df['oversold'] == os_base) & (sens_df['overbought'] == ob_base)].copy().sort_values('period')
    os_variations = sens_df[(sens_df['period'] == period_base) & (sens_df['overbought'] == ob_base)].copy().sort_values('oversold')
    ob_variations = sens_df[(sens_df['period'] == period_base) & (sens_df['oversold'] == os_base)].copy().sort_values('overbought')
    
    # Calculate deltas
    period_variations['is_sharpe_delta'] = ((period_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
    os_variations['is_sharpe_delta'] = ((os_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
    ob_variations['is_sharpe_delta'] = ((ob_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
    
    if not np.isnan(base_oos_sharpe) and abs(base_oos_sharpe) > 1e-6:
        period_variations['oos_sharpe_delta'] = ((period_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
        os_variations['oos_sharpe_delta'] = ((os_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
        ob_variations['oos_sharpe_delta'] = ((ob_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
    
    # ============================================================
    # 6-PANEL BAR CHART
    # ============================================================
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle(f'Parameter Sensitivity Analysis — Base: RSI({period_base}, OS={os_base}, OB={ob_base})', 
                 fontsize=16, fontweight='bold')
    
    configs = [
        ('RSI Period', period_variations, 'period', period_base),
        ('Oversold', os_variations, 'oversold', os_base),
        ('Overbought', ob_variations, 'overbought', ob_base),
    ]
    
    for col_idx, (param_name, var_df, param_col, base_val) in enumerate(configs):
        # IS row
        colors_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                     for x in var_df['is_sharpe_delta']]
        axes[0, col_idx].bar(var_df[param_col], var_df['is_sharpe_delta'], color=colors_is, edgecolor='black', linewidth=0.5)
        axes[0, col_idx].axhline(0, color='black', linewidth=1.5)
        axes[0, col_idx].axvline(base_val, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={base_val}')
        axes[0, col_idx].set_xlabel(param_name, fontsize=11, fontweight='bold')
        axes[0, col_idx].set_ylabel('Sharpe Δ%', fontsize=11, fontweight='bold')
        axes[0, col_idx].set_title(f'{param_name} — In-Sample', fontsize=12, fontweight='bold')
        axes[0, col_idx].grid(axis='y', alpha=0.3)
        axes[0, col_idx].legend(fontsize=10)
        
        # OOS row
        if not np.isnan(base_oos_sharpe) and abs(base_oos_sharpe) > 1e-6 and 'oos_sharpe_delta' in var_df.columns:
            oos_d = var_df['oos_sharpe_delta'].fillna(0)
            colors_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                         for x in oos_d]
            axes[1, col_idx].bar(var_df[param_col], oos_d, color=colors_oos, edgecolor='black', linewidth=0.5)
            axes[1, col_idx].axhline(0, color='black', linewidth=1.5)
            axes[1, col_idx].axvline(base_val, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={base_val}')
            axes[1, col_idx].set_xlabel(param_name, fontsize=11, fontweight='bold')
            axes[1, col_idx].set_ylabel('Sharpe Δ%', fontsize=11, fontweight='bold')
            axes[1, col_idx].set_title(f'{param_name} — Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, col_idx].grid(axis='y', alpha=0.3)
            axes[1, col_idx].legend(fontsize=10)
        else:
            axes[1, col_idx].text(0.5, 0.5, 'OOS Sharpe ≈ 0 or N/A', transform=axes[1, col_idx].transAxes,
                                  ha='center', va='center', fontsize=14)
            axes[1, col_idx].set_title(f'{param_name} — Out-of-Sample', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # SENSITIVITY SUMMARY TABLE
    # ============================================================
    print("\n📋 SENSITIVITY SUMMARY")
    print("=" * 80)
    
    summary_data = []
    for param_name, variations, param_col in [('RSI Period', period_variations, 'period'), 
                                               ('Oversold', os_variations, 'oversold'), 
                                               ('Overbought', ob_variations, 'overbought')]:
        is_max_delta = variations['is_sharpe_delta'].abs().max()
        summary_data.append({
            'Parameter': param_name,
            'IS Range': f"{variations['is_sharpe'].min():.3f} — {variations['is_sharpe'].max():.3f}",
            'IS Max Δ%': f"{variations['is_sharpe_delta'].min():.1f}%",
            'OOS Range': f"{variations['oos_sharpe'].min():.3f} — {variations['oos_sharpe'].max():.3f}" if not np.isnan(base_oos_sharpe) else 'N/A',
            'OOS Max Δ%': f"{variations['oos_sharpe_delta'].min():.1f}%" if not np.isnan(base_oos_sharpe) and 'oos_sharpe_delta' in variations else 'N/A',
            'Sensitivity': '⚠️ HIGH' if is_max_delta > 40 else '✅ LOW'
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    print("\n✅ Analysis Complete! Green = Robust, Red = Fragile")

---
## Section 9: Best Strategy Summary & Equity Curves

In [ ]:
# BEST STRATEGY SUMMARY WITH EQUITY CURVES & RSI PLOT

results_df = pd.DataFrame(grid_search_results)
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
p_best = int(best['rsi_period'])
os_best = int(best['oversold'])
ob_best = int(best['overbought'])

print("=" * 70)
print("🏆 BEST RSI STRATEGY SUMMARY")
print("=" * 70)
print(f"\nParameters: RSI({p_best}, OS={os_best}, OB={ob_best})")

# Run on full data
entries_full, exits_full = compute_rsi_signals(close, p_best, os_best, ob_best)
pf_full = vbt.Portfolio.from_signals(
    close=close, entries=entries_full, exits=exits_full,
    init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D'
)

# IS
entries_is, exits_is = compute_rsi_signals(train_close, p_best, os_best, ob_best)
pf_is = vbt.Portfolio.from_signals(
    close=train_close, entries=entries_is, exits=exits_is,
    init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D'
)

# OOS
entries_oos, exits_oos = compute_rsi_signals(val_close, p_best, os_best, ob_best)
pf_oos = vbt.Portfolio.from_signals(
    close=val_close, entries=entries_oos, exits=exits_oos,
    init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D'
)

print(f"\n{'Metric':<25} {'In-Sample':>15} {'Out-of-Sample':>15}")
print("-" * 55)
print(f"{'Sharpe Ratio':<25} {pf_is.sharpe_ratio(freq='D'):>15.3f} {pf_oos.sharpe_ratio(freq='D'):>15.3f}")
print(f"{'Sortino Ratio':<25} {pf_is.sortino_ratio(freq='D'):>15.3f} {pf_oos.sortino_ratio(freq='D'):>15.3f}")
print(f"{'Total Return':<25} {pf_is.total_return():>14.2%} {pf_oos.total_return():>14.2%}")
print(f"{'Annualized Return':<25} {pf_is.annualized_return(freq='D'):>14.2%} {pf_oos.annualized_return(freq='D'):>14.2%}")
print(f"{'Max Drawdown':<25} {pf_is.max_drawdown():>14.2%} {pf_oos.max_drawdown():>14.2%}")
print(f"{'Total Trades':<25} {len(pf_is.trades):>15d} {len(pf_oos.trades):>15d}")

for label, pf in [('IS', pf_is), ('OOS', pf_oos)]:
    tr = pf.trades.returns.values
    wr = (len(tr[tr > 0]) / len(tr) * 100) if len(tr) > 0 else 0
    pos_g = tr[tr > 0].sum() if len(tr[tr > 0]) else 0
    neg_l = abs(tr[tr < 0].sum()) if len(tr[tr < 0]) else 0
    pf_val = pos_g / neg_l if neg_l > 0 else np.inf
    if label == 'IS':
        print(f"{'Win Rate':<25} {wr:>14.1f}%", end='')
        is_wr, is_pf_val = wr, pf_val
    else:
        print(f" {wr:>14.1f}%")
        print(f"{'Profit Factor':<25} {is_pf_val:>15.2f} {pf_val:>15.2f}")

sharpe_decay = ((pf_oos.sharpe_ratio(freq='D') - pf_is.sharpe_ratio(freq='D')) / 
                abs(pf_is.sharpe_ratio(freq='D')) * 100)
print(f"\nSharpe Decay IS→OOS: {sharpe_decay:.1f}%")
print(f"Robustness: {'✅ ROBUST' if abs(sharpe_decay) < 30 else '⚠️ FRAGILE'}")

# ============================================================
# EQUITY CURVE (IS vs OOS)
# ============================================================
ret_full = pf_full.returns()
eq_full = (1 + ret_full).cumprod() * 100_000
split_date = train_close.index[-1]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(eq_full.index, eq_full.values, color='black', linewidth=1.5, label='Equity Curve')
ax1.axvline(split_date, color='red', linestyle='--', linewidth=2, alpha=0.8, label='IS/OOS Split')
ax1.fill_between(eq_full.index, eq_full.values, alpha=0.1, color='blue')
ax1.axvspan(eq_full.index[0], split_date, alpha=0.05, color='green', label='In-Sample')
ax1.axvspan(split_date, eq_full.index[-1], alpha=0.05, color='orange', label='Out-of-Sample')
ax1.set_title(f'RSI({p_best}, OS={os_best}, OB={ob_best}) — Equity Curve (IS vs OOS)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

peak = eq_full.cummax()
dd = (eq_full - peak) / peak
ax2.fill_between(dd.index, dd.values * 100, color='red', alpha=0.4)
ax2.axvline(split_date, color='red', linestyle='--', linewidth=2, alpha=0.8)
ax2.set_ylabel('Drawdown %', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.set_title('Underwater Equity (Drawdowns)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# RSI INDICATOR SUBPLOT WITH SIGNALS
# ============================================================
print("\n--- RSI Signal Overlay on Price ---")

rsi_full_plot = talib.RSI(close.values.astype(float), timeperiod=p_best)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

# Price + signals
ax1.plot(close.index, close.values, color='black', linewidth=1, label='Close')
entry_mask = entries_full.fillna(False)
exit_mask = exits_full.fillna(False)
ax1.scatter(close.index[entry_mask], close[entry_mask], marker='^', color='green',
            s=60, zorder=5, label='Long Entry (RSI > OS)')
ax1.scatter(close.index[exit_mask], close[exit_mask], marker='v', color='red',
            s=60, zorder=5, label='Long Exit (RSI < OB)')
ax1.axvline(split_date, color='red', linestyle='--', linewidth=2, alpha=0.7, label='IS/OOS Split')
ax1.set_title(f'{TICKER} Price with RSI({p_best}) Signals (OS={os_best}, OB={ob_best})', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# RSI panel
ax2.plot(close.index, rsi_full_plot, color='purple', linewidth=1, label=f'RSI({p_best})')
ax2.axhline(ob_best, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label=f'OB={ob_best}')
ax2.axhline(os_best, color='green', linestyle='--', linewidth=1.2, alpha=0.7, label=f'OS={os_best}')
ax2.axhline(50, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
ax2.fill_between(close.index, ob_best, 100, alpha=0.08, color='red')
ax2.fill_between(close.index, 0, os_best, alpha=0.08, color='green')
ax2.axvline(split_date, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ RSI Strategy Notebook Complete!")
print("\nNext Steps:")
print("  1. Test on FTMO assets: EURUSD=X, GBPUSD=X, JPY=X (forex)")
print("  2. Test on indices: ^DJI (US30), ^IXIC (NAS100), ^GSPC (SPX500)")
print("  3. Test on gold: GC=F (XAUUSD)")
print("  4. Change TICKER variable and re-run entire notebook")

---
## Section 10: Trade Distributions & Overall Equity Curve

Visual analysis of per-trade return distributions and a clean full-period equity curve with receipt-style performance summary including FTMO challenge metrics.

In [ ]:
# TRADE RETURN DISTRIBUTIONS

# Get trade-level data from the full-period backtest
trades_obj = pf_full.trades
trade_returns = trades_obj.returns.values
trade_pnl = trades_obj.pnl.values

wins = trade_returns[trade_returns > 0]
losses = trade_returns[trade_returns <= 0]

# Try to get durations
try:
    durations = trades_obj.duration
    if hasattr(durations, 'values'):
        dur_days = durations.dt.total_seconds().values / 86400 if hasattr(durations.iloc[0], 'total_seconds') else durations.values
    else:
        dur_days = np.array([float(d.total_seconds() / 86400) if hasattr(d, 'total_seconds') else float(d) for d in durations])
except:
    dur_days = None

print(f"Trade Statistics for RSI({p_best}, OS={os_best}, OB={ob_best}) — Full Period")
print("=" * 60)
print(f"Total Trades:    {len(trade_returns)}")
print(f"Winners:         {len(wins)} ({len(wins)/len(trade_returns)*100:.1f}%)")
print(f"Losers:          {len(losses)} ({len(losses)/len(trade_returns)*100:.1f}%)")
print(f"Mean Return:     {trade_returns.mean():.4f} ({trade_returns.mean()*100:.2f}%)")
print(f"Median Return:   {np.median(trade_returns):.4f}")
print(f"Std Dev:         {trade_returns.std():.4f}")
print(f"Skewness:        {float(pd.Series(trade_returns).skew()):.3f}")
print(f"Kurtosis:        {float(pd.Series(trade_returns).kurtosis()):.3f}")
print(f"Best Trade:      {trade_returns.max():.4f} ({trade_returns.max()*100:.2f}%)")
print(f"Worst Trade:     {trade_returns.min():.4f} ({trade_returns.min()*100:.2f}%)")
if len(wins) > 0 and len(losses) > 0:
    print(f"Avg Win:         {wins.mean():.4f} ({wins.mean()*100:.2f}%)")
    print(f"Avg Loss:        {losses.mean():.4f} ({losses.mean()*100:.2f}%)")
    print(f"Payoff Ratio:    {abs(wins.mean() / losses.mean()):.2f}x")
if dur_days is not None:
    print(f"Avg Duration:    {np.nanmean(dur_days):.1f} days")
    print(f"Median Duration: {np.nanmedian(dur_days):.1f} days")

# ============================================================
# 4-PANEL TRADE DISTRIBUTION FIGURE
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'Trade Distributions — RSI({p_best}, OS={os_best}, OB={ob_best})', 
             fontsize=16, fontweight='bold')

# Panel 1: Histogram of trade returns with KDE
ax = axes[0, 0]
n_bins = min(50, max(15, len(trade_returns) // 3))
n_ret, bins_ret, patches_ret = ax.hist(trade_returns * 100, bins=n_bins, 
                                        edgecolor='black', linewidth=0.5, alpha=0.75)
# Color bars: green for positive, red for negative
for patch, left_edge in zip(patches_ret, bins_ret[:-1]):
    if left_edge >= 0:
        patch.set_facecolor('#2ecc71')
    else:
        patch.set_facecolor('#e74c3c')
ax.axvline(0, color='black', linewidth=1.5, linestyle='-')
ax.axvline(trade_returns.mean() * 100, color='blue', linewidth=2, linestyle='--', 
           label=f'Mean = {trade_returns.mean()*100:.2f}%')
ax.axvline(np.median(trade_returns) * 100, color='orange', linewidth=2, linestyle=':', 
           label=f'Median = {np.median(trade_returns)*100:.2f}%')
ax.set_xlabel('Trade Return (%)', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_title('Distribution of Per-Trade Returns', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Panel 2: Win/Loss breakdown (grouped bar or stacked)
ax = axes[0, 1]
if len(wins) > 0 and len(losses) > 0:
    categories = ['Count', 'Avg Return %', 'Total PnL $']
    win_vals = [len(wins), wins.mean() * 100, trade_pnl[trade_returns > 0].sum()]
    loss_vals = [len(losses), abs(losses.mean()) * 100, abs(trade_pnl[trade_returns <= 0].sum())]
    
    x = np.arange(len(categories))
    width = 0.35
    bars1 = ax.bar(x - width/2, win_vals, width, label='Winners', color='#2ecc71', edgecolor='black', linewidth=0.5)
    bars2 = ax.bar(x + width/2, loss_vals, width, label='Losers', color='#e74c3c', edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=10, fontweight='bold')
    # Add value labels on bars
    for bar in bars1:
        h = bar.get_height()
        fmt = f'{h:.0f}' if h > 10 else f'{h:.2f}'
        ax.text(bar.get_x() + bar.get_width()/2., h, fmt, ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar in bars2:
        h = bar.get_height()
        fmt = f'{h:.0f}' if h > 10 else f'{h:.2f}'
        ax.text(bar.get_x() + bar.get_width()/2., h, fmt, ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Winners vs Losers Breakdown', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Panel 3: Cumulative PnL per trade (trade sequence)
ax = axes[1, 0]
cum_pnl = np.cumsum(trade_pnl)
trade_nums = np.arange(1, len(trade_pnl) + 1)
ax.plot(trade_nums, cum_pnl, color='black', linewidth=1.5)
ax.fill_between(trade_nums, cum_pnl, where=(cum_pnl >= 0), color='#2ecc71', alpha=0.3)
ax.fill_between(trade_nums, cum_pnl, where=(cum_pnl < 0), color='#e74c3c', alpha=0.3)
ax.axhline(0, color='gray', linewidth=1, linestyle='--')
ax.set_xlabel('Trade Number', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative PnL ($)', fontsize=11, fontweight='bold')
ax.set_title('Cumulative PnL by Trade Sequence', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Panel 4: Trade duration distribution (if available)
ax = axes[1, 1]
if dur_days is not None and len(dur_days) > 0:
    valid_dur = dur_days[~np.isnan(dur_days)]
    if len(valid_dur) > 0:
        n_bins_d = min(40, max(10, len(valid_dur) // 3))
        ax.hist(valid_dur, bins=n_bins_d, color='#3498db', edgecolor='black', linewidth=0.5, alpha=0.75)
        ax.axvline(np.nanmean(valid_dur), color='blue', linewidth=2, linestyle='--', 
                   label=f'Mean = {np.nanmean(valid_dur):.1f}d')
        ax.axvline(np.nanmedian(valid_dur), color='orange', linewidth=2, linestyle=':', 
                   label=f'Median = {np.nanmedian(valid_dur):.1f}d')
        ax.set_xlabel('Trade Duration (days)', fontsize=11, fontweight='bold')
        ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
        ax.set_title('Distribution of Trade Durations', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(axis='y', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Duration data unavailable', transform=ax.transAxes, ha='center', va='center', fontsize=14)
else:
    ax.text(0.5, 0.5, 'Duration data unavailable', transform=ax.transAxes, ha='center', va='center', fontsize=14)
    ax.set_title('Distribution of Trade Durations', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# OVERALL EQUITY CURVE WITH RECEIPT-STYLE STATS BOX + FTMO METRICS

# ============================================================
# Compute all stats for the receipt
# ============================================================
full_total_return = float(pf_full.total_return())
full_ann_return = float(pf_full.annualized_return(freq='D'))
full_sharpe = float(pf_full.sharpe_ratio(freq='D'))
full_sortino = float(pf_full.sortino_ratio(freq='D'))
full_max_dd = float(pf_full.max_drawdown())
full_volatility = float(pf_full.annualized_volatility(freq='D'))
full_total_trades = len(pf_full.trades)

# CAGR
years_full = (close.index[-1] - close.index[0]).days / 365.25
full_cagr = (1 + full_total_return) ** (1 / max(years_full, 1e-9)) - 1

# Trade stats
tr_full = pf_full.trades.returns.values
full_win_rate = (len(tr_full[tr_full > 0]) / len(tr_full) * 100) if len(tr_full) > 0 else 0
pos_g = tr_full[tr_full > 0].sum() if len(tr_full[tr_full > 0]) else 0
neg_l = abs(tr_full[tr_full < 0].sum()) if len(tr_full[tr_full < 0]) else 0
full_pf = pos_g / neg_l if neg_l > 0 else np.inf
full_trades_yr = full_total_trades / max(years_full, 1e-9)

# FTMO-specific metrics
# Max daily drawdown (worst single day loss)
daily_returns = pf_full.returns()
max_daily_loss = float(daily_returns.min())  # worst single day

# Rolling max drawdown (worst peak-to-trough in portfolio value)
eq_curve = (1 + daily_returns).cumprod() * 100_000
running_peak = eq_curve.cummax()
drawdown_series = (eq_curve - running_peak) / running_peak

# Consecutive losing streak
streak = 0
max_streak = 0
for r in tr_full:
    if r <= 0:
        streak += 1
        max_streak = max(max_streak, streak)
    else:
        streak = 0

# FTMO challenge pass/fail estimates
ftmo_profit_target_p1 = 0.10  # 10% for Phase 1
ftmo_profit_target_p2 = 0.05  # 5% for Phase 2
ftmo_max_daily_loss = 0.05    # 5% max daily loss
ftmo_max_overall_loss = 0.10  # 10% max overall loss

passes_daily_dd = abs(max_daily_loss) < ftmo_max_daily_loss
passes_overall_dd = abs(full_max_dd) < ftmo_max_overall_loss
passes_profit_p1 = full_cagr >= ftmo_profit_target_p1

# ============================================================
# PLOT: Overall Equity Curve with Receipt Box
# ============================================================
fig, ax_main = plt.subplots(1, 1, figsize=(18, 10))

# Equity curve
ax_main.plot(eq_curve.index, eq_curve.values, color='#1a1a2e', linewidth=2, label='Portfolio Equity')
ax_main.fill_between(eq_curve.index, eq_curve.values, eq_curve.values.min() * 0.95,
                     alpha=0.08, color='#3498db')

# Add drawdown shading
dd_pct = drawdown_series * 100

# Style the chart
ax_main.set_title(f'{TICKER} — RSI({p_best}, OS={os_best}, OB={ob_best}) Overall Equity Curve', 
                  fontsize=16, fontweight='bold', pad=15)
ax_main.set_ylabel('Portfolio Value ($)', fontsize=13, fontweight='bold')
ax_main.set_xlabel('Date', fontsize=13, fontweight='bold')
ax_main.set_yscale('log')
ax_main.grid(True, alpha=0.3)
ax_main.legend(loc='upper left', fontsize=11)

# ============================================================
# RECEIPT-STYLE STATS BOX (top-right corner)
# ============================================================
receipt_lines = [
    '┌──────────────────────────────┐',
    '│    STRATEGY PERFORMANCE       │',
    '│         RECEIPT               │',
    '├──────────────────────────────┤',
    f'│  Ticker:       {TICKER:<14s}│',
    f'│  Strategy: RSI {p_best}/{os_best}/{ob_best}' + ' '*(16-len(f'{p_best}/{os_best}/{ob_best}')) + '│',
    '├──────────────────────────────┤',
    '│  RETURNS                     │',
    f'│  Total Return: {full_total_return:>+12.2%}  │',
    f'│  CAGR:         {full_cagr:>+12.2%}  │',
    f'│  Ann. Return:  {full_ann_return:>+12.2%}  │',
    '├──────────────────────────────┤',
    '│  RISK-ADJUSTED               │',
    f'│  Sharpe Ratio: {full_sharpe:>12.3f}  │',
    f'│  Sortino Ratio:{full_sortino:>12.3f}  │',
    f'│  Volatility:   {full_volatility:>11.2%}  │',
    '├──────────────────────────────┤',
    '│  RISK                        │',
    f'│  Max Drawdown: {full_max_dd:>11.2%}  │',
    f'│  Max Daily Loss:{max_daily_loss:>10.2%}  │',
    f'│  Max Loss Streak:{max_streak:>9d}  │',
    '├──────────────────────────────┤',
    '│  TRADES                      │',
    f'│  Total Trades: {full_total_trades:>12d}  │',
    f'│  Trades/Year:  {full_trades_yr:>12.1f}  │',
    f'│  Win Rate:     {full_win_rate:>11.1f}%  │',
    f'│  Profit Factor:{full_pf:>12.2f}  │',
    '├──────────────────────────────┤',
    '│  FTMO CHALLENGE              │',
    f'│  Daily DD < 5%:  {"✅ PASS" if passes_daily_dd else "❌ FAIL":>10s}  │',
    f'│  Overall DD<10%: {"✅ PASS" if passes_overall_dd else "❌ FAIL":>10s}  │',
    f'│  CAGR > 10%:     {"✅ PASS" if passes_profit_p1 else "❌ FAIL":>10s}  │',
    '└──────────────────────────────┘',
]

receipt_text = '\n'.join(receipt_lines)

# Place the receipt box
ax_main.text(
    0.98, 0.02, receipt_text,
    transform=ax_main.transAxes,
    fontsize=9,
    fontfamily='monospace',
    verticalalignment='bottom',
    horizontalalignment='right',
    bbox=dict(
        boxstyle='round,pad=0.6',
        facecolor='#fafafa',
        edgecolor='#333333',
        linewidth=1.5,
        alpha=0.95
    )
)

plt.tight_layout()
plt.show()

# ============================================================
# PRINT FTMO SUMMARY
# ============================================================
print("\n" + "=" * 50)
print("FTMO CHALLENGE READINESS")
print("=" * 50)
print(f"Max Daily Loss:      {max_daily_loss:.2%}  (Limit: -5.00%)  {'✅' if passes_daily_dd else '❌'}")
print(f"Max Overall DD:      {full_max_dd:.2%}  (Limit: -10.00%) {'✅' if passes_overall_dd else '❌'}")
print(f"CAGR:                {full_cagr:.2%}  (Target: +10.00%) {'✅' if passes_profit_p1 else '❌'}")
print(f"Max Losing Streak:   {max_streak} trades")
print(f"\nOverall Assessment:  {'✅ FTMO VIABLE' if (passes_daily_dd and passes_overall_dd) else '⚠️ REVIEW RISK LIMITS'}")